# 03 · Blind judge file and final score
No GPU needed.
1. Build `judge_items.csv` (both systems mixed, shuffled, system names hidden, 10% hidden duplicates).
2. Send **only** `judge_items.csv` to the judge (Claude). Keep `judge_key.csv` private.
3. Put the returned `judge_scores.csv` in the same folder and run the scoring cell.

In [ ]:
# ---- 1. Settings (edit once) ----
REPO_URL   = "https://github.com/O-sama2004/gemma-tr-temporal-inference.git"
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
DATA_VERSION = "v1"
DATA = "/content/data"          # local copy (Drive I/O is slow)

In [ ]:
import os, subprocess, shutil
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
os.chdir("/content/repo")
!pip install -q sacrebleu==2.5.1
shutil.copytree(f"{DRIVE_ROOT}/data/{DATA_VERSION}", DATA, dirs_exist_ok=True)
!python src/verify_data.py {DATA}

## Step 1: build the judge file

In [ ]:
RUN_ID = "PASTE_RUN_ID_HERE"        # folder name under runs/
JUDGE = f"{DRIVE_ROOT}/eval/judge_{RUN_ID}"
BASE = f"{DRIVE_ROOT}/eval/baseline_untrained/predictions_test.csv"
FT = f"{DRIVE_ROOT}/runs/{RUN_ID}/eval/predictions_test.csv"
!python src/make_judge_file.py --data_dir {DATA} --pred baseline="{BASE}" --pred finetuned="{FT}" --out_dir "{JUDGE}"
print("send this file to the judge:", f"{JUDGE}/judge_items.csv")

## Step 2: after `judge_scores.csv` is in the folder

In [ ]:
!python src/score.py --judge_dir "{JUDGE}" --scores "{JUDGE}/judge_scores.csv" --baseline baseline --compare finetuned